# Runda active learning: odpowiedzi analityka wracają do modelu

Domyka pętlę. Analityk wypełnia `kolejka.csv` z `analyst/do_oznaczenia/<partia>/` i odkłada go do `analyst/oznaczone/<partia>/`. Ten notebook uruchamia rundę: mały pipeline z jednym krokiem.

1. Odpowiedzi analityka dochodzą do wzorców typologii od razu, bez trenowania.
2. Bramka porównuje nowe wzorce z tymi w użyciu na zbiorze kontrolnym. Przyjęte idą do użycia (`analyst/wzorce/<przebieg>/fold<k>/produkcja.json`), a przy odrzuceniu zostają stare wzorce i odpowiedzi czekają na drugą weryfikację.
3. Powstaje następna partia `<partia>-r1` (potem `-r2` itd.) z raportem rundy. Analityk dostaje maila z regułą „FSL - runda active learning”.

Kolejność: komórki 1–2, komórka 3 tylko na demo (zamiast analityka), potem 4–5. Komórka 6 pokazuje wynik po zakończeniu przebiegu. Kolejna runda zaczyna się znowu od komórki 2.

**Z triggerem** (`deploy_trigger.sh`, funkcja Cloud Run) runda startuje sama, gdy w `analyst/oznaczone/<partia>/` pojawi się `kolejka.csv`, czyli po komórce 3 albo po uploadzie analityka. Komórki 5 wtedy nie uruchamiasz. Komórkę 4 trzeba uruchomić raz (i po każdej zmianie pipeline'u rundy), bo zapisuje w GCS szablon, z którego korzysta trigger.

Wymagania:
- obraz przebudowany z nowymi `scripts/al_simulation_entry.py` i `scripts/al_round_entry.py`;
- partia z przebiegu na tym obrazie, bo runda potrzebuje jej `stan.json`;
- trzecia reguła maila: komórki 2–4 w `analyst_alert_setup.ipynb` uruchomione jeszcze raz.

In [ ]:
# KOMORKA 1 — config
import io
import json
from pathlib import Path

import numpy as np
import pandas as pd
from google.cloud import storage

REPO = Path.home() / "mlops-meta-learning/fsl-projekt"
CFG = json.loads((REPO / "configs/pipeline_config_samld.json").read_text())
PROJECT, BUCKET, REGION = CFG["project"], CFG["bucket"], CFG.get("region", "us-central1")
IMAGE = CFG["training_image_uri"]
ANALYST_EMAILS = CFG.get("notifications", {}).get("analyst_emails", [])
assert ANALYST_EMAILS, "brak notifications.analyst_emails w configu"
CONSOLE_ACCOUNT = CFG.get("notifications", {}).get("console_account", ANALYST_EMAILS[0])
gcs = storage.Client(project=PROJECT)
bkt = gcs.bucket(BUCKET)
A = "analyst"
print("projekt:", PROJECT, "| obraz:", IMAGE)

In [ ]:
# KOMORKA 2 — partie dla analityka i ich stan. BATCH = najnowsza partia bez zrobionej rundy.
rows = []
for b in gcs.list_blobs(BUCKET, prefix=f"{A}/do_oznaczenia/"):
    if not b.name.endswith("/stan.json"):
        continue
    batch = b.name[len(f"{A}/do_oznaczenia/"):-len("/stan.json")]
    if bkt.blob(f"{A}/oznaczone/{batch}/wynik.json").exists():
        status = "runda zrobiona"
    elif bkt.blob(f"{A}/oznaczone/{batch}/kolejka.csv").exists():
        status = "wypelniona, gotowa do rundy"
    else:
        status = "czeka na analityka"
    rows.append((b.updated, batch, status))
rows.sort(reverse=True)
for upd, batch, status in rows[:15]:
    print(f"{upd:%Y-%m-%d %H:%M}  {batch:<60} {status}")
if not rows:
    print("Brak partii ze stan.json: pusc pipeline z enable_al=True albo kampanie na przebudowanym obrazie.")
BATCH = next((b for _, b, s in rows if s != "runda zrobiona"), None)
# Inna partia: wpisz ja recznie, np.
# BATCH = "fsl-samld-case-campaign-20261006123456/fold3"
print("\nBATCH =", BATCH)

In [ ]:
# KOMORKA 3 — TYLKO NA DEMO: "analityk" wpisuje prawdziwe typologie z danych i odklada plik do oznaczone/.
# Z prawdziwym analitykiem pomin te komorke: on sam odklada wypelniony kolejka.csv.
# POMYLKI = ile odpowiedzi (z 8) celowo zlych, zeby pokazac bramke. Na danych testowych bramka odrzucala
# rundy z 3 zlymi odpowiedziami mniej wiecej co drugi raz, z 8 zlymi w ok. 85% przypadkow.
POMYLKI = 0

from fsl.config import TrainConfig
from fsl.data.registry import get_loaders

stan = json.loads(bkt.blob(f"{A}/do_oznaczenia/{BATCH}/stan.json").download_as_text())
cfg = TrainConfig(project=PROJECT, bucket=BUCKET, dataset=stan["dataset"], seed=0, fold=stan["fold"],
                  n_way=4, k_shot=5, query=5, log_to_vertex=False)
load_frozen, _ = get_loaders(cfg)
bundle, _ = load_frozen(cfg)
sch, W = bundle["schema"], bundle["windows"]
truth = dict(zip(W[sch["id_col"]].astype(int), W[sch["label_col"]].astype(str)))

q = pd.read_csv(io.BytesIO(bkt.blob(f"{A}/do_oznaczenia/{BATCH}/kolejka.csv").download_as_bytes()),
                sep=";", encoding="utf-8-sig", dtype=str, keep_default_na=False)
q["typologia_analityka"] = [truth[int(c)] for c in q["przypadek"]]
rng = np.random.default_rng(0)
for i in rng.choice(len(q), min(POMYLKI, len(q)), replace=False):
    q.loc[i, "typologia_analityka"] = rng.choice([n for n in stan["names"] if n != q.loc[i, "typologia_analityka"]])
    q.loc[i, "komentarz"] = "celowa pomylka (demo)"
bkt.blob(f"{A}/oznaczone/{BATCH}/kolejka.csv").upload_from_string(
    q.to_csv(index=False, sep=";").encode("utf-8-sig"), content_type="text/csv; charset=utf-8")
print(q[["kolejnosc", "przypadek", "podpowiedz_modelu", "typologia_analityka", "komentarz"]].to_string(index=False))
print(f"\nanalityk potwierdzil {int((q.podpowiedz_modelu == q.typologia_analityka).sum())} z {len(q)} podpowiedzi;",
      f"plik w gs://{BUCKET}/{A}/oznaczone/{BATCH}/kolejka.csv")

In [ ]:
# KOMORKA 4 — pipeline rundy: jeden krok + mail od Vertex na koniec (jak w pozostalych pipeline'ach)
from kfp import compiler, dsl
from kfp.dsl import HTML, ContainerSpec, Output, OutputPath
from google_cloud_pipeline_components.v1.vertex_notification_email import VertexNotificationEmailOp


@dsl.container_component
def al_round_container(
    project: str,
    region: str,
    bucket: str,
    batch: str,
    run_name: str,
    queue_size: int,
    max_drop: float,
    report: Output[HTML],
    summary: OutputPath(str),
):
    return ContainerSpec(
        image=IMAGE,
        command=["python", "scripts/al_round_entry.py"],
        args=[
            "--project", project, "--region", region, "--bucket", bucket,
            "--batch", batch, "--run-name", run_name,
            "--queue-size", queue_size, "--max-drop", max_drop,
            "--report-html-path", report.path,
            "--summary-output-path", summary,
        ],
    )


@dsl.pipeline(name="fsl-al-round")
def al_round_pipeline(
    batch: str,
    project: str = PROJECT,
    region: str = REGION,
    bucket: str = BUCKET,
    queue_size: int = 8,
    max_drop: float = 0.02,
):
    notify = VertexNotificationEmailOp(recipients=ANALYST_EMAILS)
    with dsl.ExitHandler(notify, name="mail-do-analityka"):
        (al_round_container(project=project, region=region, bucket=bucket, batch=batch,
                            run_name=dsl.PIPELINE_JOB_NAME_PLACEHOLDER,
                            queue_size=queue_size, max_drop=max_drop)
         .set_caching_options(False)
         .set_display_name("runda-active-learning"))


compiler.Compiler().compile(al_round_pipeline, "al_round_pipeline.yaml")
# kopia w GCS: z niej korzysta trigger (funkcja Cloud Run uruchamia runde po wrzuceniu pliku do oznaczone/)
bkt.blob("pipelines/al_round_pipeline.yaml").upload_from_filename("al_round_pipeline.yaml")
print(f"skompilowany: al_round_pipeline.yaml, kopia dla triggera: gs://{BUCKET}/pipelines/al_round_pipeline.yaml")

In [ ]:
# KOMORKA 5 — uruchom runde RECZNIE dla BATCH (ok. 5 minut; potem przychodzi mail z wynikiem i nastepna partia).
# Z wdrozonym triggerem (deploy_trigger.sh) runda startuje sama po wrzuceniu pliku do oznaczone/ —
# wtedy tej komorki NIE uruchamiaj, bo runda poszlaby dwa razy.
from google.cloud import aiplatform

assert BATCH, "BATCH nie ustawiony (komorka 2)"
assert bkt.blob(f"{A}/oznaczone/{BATCH}/kolejka.csv").exists(), \
    f"brak wypelnionego pliku gs://{BUCKET}/{A}/oznaczone/{BATCH}/kolejka.csv"
aiplatform.init(project=PROJECT, location=REGION)
job = aiplatform.PipelineJob(
    display_name="fsl-al-round",
    template_path="al_round_pipeline.yaml",
    pipeline_root=f"gs://{BUCKET}/pipeline-root",
    parameter_values={"batch": BATCH},
    enable_caching=False,
)
job.submit()
print("runda uruchomiona dla partii", BATCH)

In [ ]:
# KOMORKA 6 — wynik rundy (po zakonczeniu przebiegu)
w = json.loads(bkt.blob(f"{A}/oznaczone/{BATCH}/wynik.json").download_as_text())
print(f"runda {w['round']}: trafnosc {100 * w['accuracy_in_use_before']:.1f}% -> {100 * w['accuracy_new']:.1f}% | "
      f"bramka: {'PRZYJETE' if w['accepted'] else 'ODRZUCONE'} | etykiet w uzyciu: {w['labels_in_use']} | "
      f"analityk poprawil model w {w['analyst_fixed']} z {w['new_labels']}")
if w["held_for_review"]:
    print("czekaja na druga weryfikacje:", w["held_for_review"])
sep = "&" if "?" in w["report"] else "?"
print("raport rundy:", f"{w['report']}{sep}authuser={CONSOLE_ACCOUNT}" if w["report"].startswith("http") else w["report"])
print("nastepna partia:", w["next_batch"] or "brak (pula wyczerpana)")

**Następna runda:** komórka 2 (BATCH ustawi się na nową partię `-r1`), potem komórka 3 na demo albo plik od analityka, potem komórka 5. Komórki 4 nie trzeba powtarzać w tej samej sesji.

**Na demo:** rundy puść wcześniej, a na demo pokaż gotowe raporty i maile. Runda trwa kilka minut, a jej wynik zależy od danych. Dobra sekwencja to 2–3 rundy z `POMYLKI = 0`, które dają wykres trafności po rundach, i jedna z `POMYLKI = 8`, w której bramka zwykle odrzuca wzorce. Jeśli bramka przepuści, puść kolejną.

Bramka pilnuje trafności wzorców, a nie poprawności każdej odpowiedzi, więc nie jest wykrywaczem błędów analityka:
- **Złe odpowiedzi może przepuścić,** jeśli trafność spadnie mniej niż o 2 pp.
- **Czasem odrzuca poprawne odpowiedzi.** Wzorzec to średnia, a przypadki z kolejki leżą na granicy typologii, więc mogą ją przesunąć w złą stronę. Na danych testowych zdarzało się to mniej więcej w co piątej rundzie.

Odrzucone odpowiedzi nie przepadają: czekają w `stan.json` (`held`) na decyzję człowieka.